# Graphaufbau aus `data.csv` für Neo4j

Dieses Notebook baut deinen Sensordaten-Graphen aus der Datei `data.csv` auf.

Es führt folgende Schritte aus:
1. Aufbau der Verbindung zu Neo4j
2. Anlegen von Constraints und Indexen
3. Import der Daten aus `data.csv` in einem **einzigen Durchlauf** mittels `USING PERIODIC COMMIT`
4. Erzeugung aller Knoten (`Sensor`, `Messung`, `Epoche`, `Week`, `Date`, `Hour`, `Minute`)
5. Erzeugung aller Beziehungen entsprechend deinem Modell

Die Datei `data.csv` muss im `import`-Verzeichnis deiner Neo4j-Installation liegen.

In [ ]:
from neo4j import GraphDatabase

# Verbindungsdaten zu Neo4j anpassen
uri = "neo4j://localhost:7687"  # oder bolt://localhost:7687
user = "neo4j"
password = "YOUR_PASSWORD_HERE"  # HIER dein Passwort eintragen

driver = GraphDatabase.driver(uri, auth=(user, password))
print("Verbindung zu Neo4j hergestellt.")

def run_cypher(query: str, params: dict | None = None):
    """Hilfsfunktion zum Ausführen von Cypher-Statements und Konsumieren der Ergebnisse."""
    if params is None:
        params = {}
    with driver.session() as session:
        result = session.run(query, params)
        summary = result.consume()
    return summary

def print_summary(summary):
    """Kurze Zusammenfassung für Schreib-Queries ausgeben."""
    counters = summary.counters
    print("Nodes created:", counters.nodes_created)
    print("Relationships created:", counters.relationships_created)
    print("Properties set:", counters.properties_set)
    print("Labels added:", counters.labels_added)
    print("Indexes added:", counters.indexes_added)
    print("Constraints added:", counters.constraints_added)

## 1. Constraints & Indexe anlegen

Diese Constraints und Indexe stellen sicher, dass wichtige Knoten eindeutig
identifizierbar und Abfragen performant sind.

In [ ]:
constraints_query = """
CREATE CONSTRAINT sensor_id_unique IF NOT EXISTS
FOR (s:Sensor) REQUIRE s.sensor_id IS UNIQUE;

CREATE CONSTRAINT room_id_unique IF NOT EXISTS
FOR (r:Room) REQUIRE r.room_id IS UNIQUE;

CREATE INDEX messung_moteid IF NOT EXISTS
FOR (m:Messung) ON (m.moteid);

CREATE INDEX messung_time IF NOT EXISTS
FOR (m:Messung) ON (m.time);

CREATE CONSTRAINT epoch_id_unique IF NOT EXISTS
FOR (e:Epoche) REQUIRE e.epoch_id IS UNIQUE;

CREATE CONSTRAINT date_unique IF NOT EXISTS
FOR (d:Date) REQUIRE d.date IS UNIQUE;

CREATE INDEX hour_index IF NOT EXISTS
FOR (h:Hour) ON (h.hour);

CREATE INDEX minute_index IF NOT EXISTS
FOR (m:Minute) ON (m.minute);

CREATE INDEX week_index IF NOT EXISTS
FOR (w:Week) ON (w.week);
"""

summary = run_cypher(constraints_query)
print("Constraints & Indexe gesetzt.")
print_summary(summary)

## 2. CSV-Import mit `USING PERIODIC COMMIT`

In diesem Schritt wird **data.csv nur ein einziges Mal** durchlaufen.
Für jede Zeile werden:

- `Sensor`-Knoten erzeugt/verbunden
- `Epoche`-Knoten erzeugt/verbunden
- `Week`, `Date`, `Hour`, `Minute`-Knoten erzeugt/verbunden
- ein `Messung`-Knoten erstellt
- alle relevanten Beziehungen angelegt

Die Option `USING PERIODIC COMMIT 5000` sorgt dafür, dass alle 5000 Zeilen
eine Teilschreibtransaktion abgeschlossen wird. Das erlaubt den Import von
    

Millionen von Datensätzen ohne Speicherprobleme.

In [ ]:
import_query = """
USING PERIODIC COMMIT 5000
LOAD CSV WITH HEADERS FROM 'file:///data.csv' AS row

WITH row,
     row.date AS date,
     row.date + 'T' + row.time AS timestamp,
     toInteger(row.moteid) AS moteid,
     toInteger(row.epoch) AS epoch,
     toFloat(row.temperature) AS temp,
     toFloat(row.humidity) AS hum,
     toFloat(row.light) AS light,
     toFloat(row.voltage) AS volt,
     toInteger(split(row.time, ':')[0]) AS hour,
     toInteger(split(split(row.time, ':')[1], '.')[0]) AS minute,
     date(row.date) AS d,
     date(row.date).week AS week

MERGE (s:Sensor {sensor_id: row.moteid})
MERGE (e:Epoche {epoch_id: epoch})
MERGE (w:Week {week: week})
MERGE (dt:Date {date: date})
MERGE (w)-[:HAS_DATE]->(dt)
MERGE (h:Hour {hour: hour})
MERGE (dt)-[:HAS_HOUR]->(h)
MERGE (min:Minute {minute: minute})
MERGE (h)-[:HAS_MINUTE]->(min)

CREATE (m:Messung {
  moteid: moteid,
  temperature: temp,
  humidity: hum,
  light: light,
  voltage: volt,
  time: timestamp
})

MERGE (s)-[:GENERATED]->(m)
MERGE (m)-[:EPOCH_IN]->(e)
MERGE (m)-[:AT_WEEK]->(w)
MERGE (m)-[:AT_DATE]->(dt)
MERGE (m)-[:AT_HOUR]->(h)
MERGE (m)-[:AT_MINUTE]->(min)
MERGE (e)-[:IN_MINUTE]->(min);
"""

summary = run_cypher(import_query)
print("Import abgeschlossen.")
print_summary(summary)

## 3. Abschluss

Der Graph ist jetzt aufgebaut und kann für Analysen und LLM-generierte
Cypher-Abfragen genutzt werden.